# 04 — Train baselines and sensor-ablation models on Colab

Use **Colab CPU** first. These tabular models do not require a GPU; a compatible GPU XGBoost installation is optional. Random Forest remains CPU-based. No real model is bundled. All preprocessing is fit on training inputs only; two declared model-capacity candidates use validation, and test outcomes are not used for early stopping.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, json
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path("/content/cht-soil-moisture-gapfilling")]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/cht-soil-moisture-gapfilling")
    subprocess.run(["git", "clone", "--branch", "research-notebooks-v2", "--single-branch",
                    "https://github.com/Bipul-roy-AI/cht-soil-moisture-gapfilling.git", str(ROOT)], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned repository, or run it in Colab.")
ROOT = ROOT.resolve()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee,notebooks]"], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/CHT_SM_v2")
else:
    WORK = ROOT / "work_v2"
WORK.mkdir(parents=True, exist_ok=True)
print("Persistent work folder:", WORK)


In [ ]:
from chtsm.data import read_config, load_exports, audit_frame, save_json
CONFIG_PATH = WORK / "study_config.json"
config = read_config(CONFIG_PATH)
# Move the monthly FULL exports from the Drive export folder into this folder.
# Pilot exports have a different prefix and must not be mixed into the full table.
EXPORT_DIR = WORK / "exports"
CSV_PATHS = sorted(EXPORT_DIR.glob("chtsm_v2_full_*.csv"))
if not CSV_PATHS:
    raise FileNotFoundError("Put completed monthly chtsm_v2_full_*.csv exports in WORK/exports first.")
frame = load_exports(CSV_PATHS, config)


In [ ]:
if importlib.util.find_spec("xgboost") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "xgboost>=2.1,<4"], check=True)
from chtsm.models import fit_experiment
DEVICE = "cpu"  # Optional "cuda" ONLY with a compatible GPU build/runtime.
RUN_NAME = "study_v2_temporal_001"  # New name for a deliberate configuration change.
RUN = WORK / "runs" / RUN_NAME
metadata, validation_table = fit_experiment(frame, config, RUN, device=DEVICE)
display(validation_table[["name", "validation_macro_cell_rmse", "seed"]])
print("Validation-selected reconstruction model:", metadata["selected_model"])
print("Saved models/configuration/calibration/data hashes:", RUN)

The default experiment fits RF and XGBoost for three feature groups and three seeds, plus three simple baselines. Report every seed; selecting a deployment model on validation does not justify reporting only its test seed. Existing run folders are protected. Resume during a partially completed fit is not implemented; retain the partial folder and use a new run name. Calibration uses daily maximum residual clusters; temporal exchangeability is not guaranteed and empirical test coverage must be reported.

In [ ]:
from chtsm.models import freeze_experiment
FREEZE_FOR_FINAL_TEST = False
if FREEZE_FOR_FINAL_TEST:
    note = (WORK / "protocol_note.txt").read_text()
    freeze_experiment(RUN, note)
    print("Local protocol/run snapshot frozen. Final test remains in notebook 05.")
else:
    print("Review validation choices and protocol, then explicitly enable FREEZE_FOR_FINAL_TEST.")